# Heart-rate values summary EDA
For Friday October 23, 2026. This is the sixth reviewed summary table: activity, steps, goals, kilocalories, stress and now heart_rate_values. **The heart_rate_sample table is not loaded or reviewed here.** Only heart_rate_values and activity are read; no database changes are made.

Questions: Are summary values complete and internally consistent? Which reporting periods match activity and which are absent? How do recorded average/resting values relate descriptively to activity within and across participants?

Working interpretation: min/max/avg/rest fields appear to summarize heart rate, provisionally beats per minute based on names. The `_min` suffix, averaging weights, resting algorithm and source definitions are **unverified**. SQL SMALLINT does not verify units. No source dictionary is supplied. No clinical thresholds, diagnoses, causal claims or predictive validation are used.

## 1. Inspect all fields and establish candidate grain
A synthetic example is one participant's labelled reporting window with minimum, maximum, average and resting estimates. It is not a raw beat/sample, and a date label does not prove complete-day coverage. IDs/group/type describe context; date/week/day label the record; start/end/duration describe the window. We inspect the schema and aggregate quality without displaying participant records. Credentials remain private in `.env`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately.'
engine=create_engine(url); tables={}; schema=[]
try:
    inspector=inspect(engine)
    for name in ['heart_rate_values','activity']:
        for c in inspector.get_columns(name):
            schema.append({'table':name,'column':c['name'],'sql_type':str(c['type'])})
        with engine.connect() as conn:
            tables[name]=pd.read_sql(text('SELECT * FROM '+name),conn)
finally: engine.dispose()
del url
for df in tables.values(): df['calendar_date']=pd.to_datetime(df.calendar_date,errors='coerce')
h=tables['heart_rate_values'].copy()
metrics=['min_heart_rate_min','max_heart_rate_min','avg_heart_rate_min','rest_heart_rate_min']
display(pd.DataFrame(schema).query("table == 'heart_rate_values'"))
display(pd.DataFrame({'column':h.columns,'pandas_type':h.dtypes.astype(str).values,
 'missing_rows':h.isna().sum().values,'distinct_nonnull':h.nunique().values}))
display(pd.DataFrame([{'table':name,'records':len(df),'participants':df.sdv_user_id.nunique()} for name,df in tables.items()]))


## 2. Keys, invalid/sentinel candidates and ordering
Test record ID, participant/date and participant/date/period uniqueness separately. Count nulls, zeros and negatives for every heart-rate field. Nonpositive values are flagged as operational exclusions for exploratory plots/correlations, not assigned a source-specific sentinel meaning. Positive values also cannot be certified valid without documentation.

Test min ≤ average ≤ max on complete triples, and min ≤ max on complete pairs. A resting estimate may be derived from another window or algorithm. We report whether it lies outside the reported extrema **as a diagnostic only**, not a validity requirement or medical finding. Do not impute missing records/values as zero.

In [ ]:
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
audit=[]
for name,df in tables.items():
    for label,k in [('record_id',['additional_id']),('participant_date',['sdv_user_id','calendar_date']),('participant_period',key)]:
        audit.append({'table':name,'key':label,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
          'duplicate_key_rows':int(df.duplicated(k,keep=False).sum()),'valid_unique_key':valid_key(df,k)})
display(pd.DataFrame(audit)); print('Exact duplicate rows after first:',int(h.duplicated().sum()))
display(pd.DataFrame([{'field':m,'missing':int(h[m].isna().sum()),'zero':int(h[m].eq(0).sum()),
 'negative':int(h[m].lt(0).sum()),'positive':int(h[m].gt(0).sum())} for m in metrics]))
for m in metrics:
    display(h.loc[h[m].le(0),m].value_counts().rename_axis(m).to_frame('nonpositive_value_records'))
display(h[metrics].describe().T)
triple=h[['min_heart_rate_min','avg_heart_rate_min','max_heart_rate_min']].notna().all(axis=1)
pair=h[['min_heart_rate_min','max_heart_rate_min']].notna().all(axis=1)
rest_complete=h[['rest_heart_rate_min','min_heart_rate_min','max_heart_rate_min']].notna().all(axis=1)
ordering=(h.min_heart_rate_min.le(h.avg_heart_rate_min) & h.avg_heart_rate_min.le(h.max_heart_rate_min))
display(pd.DataFrame([{'complete_min_avg_max_triples':int(triple.sum()),'excluded_incomplete_triples':int((~triple).sum()),
 'ordered_triples':int((triple & ordering).sum()),'unordered_triples':int((triple & ~ordering).sum()),
 'complete_min_max_pairs':int(pair.sum()),'min_above_max':int((pair & h.min_heart_rate_min.gt(h.max_heart_rate_min)).sum()),
 'rest_and_extrema_complete':int(rest_complete.sum()),
 'rest_below_reported_min_diagnostic':int((rest_complete & h.rest_heart_rate_min.lt(h.min_heart_rate_min)).sum()),
 'rest_above_reported_max_diagnostic':int((rest_complete & h.rest_heart_rate_min.gt(h.max_heart_rate_min)).sum())}]))


## 3. Reporting periods, overlap and coverage
Check nonpositive periods, end-minus-start consistency, hours and overlap with the maximum preceding end for the same participant. Equal endpoints are not counted as overlap here. Window duration is not wear time. Unix-seconds/UTC conversion is a plausibility check with timezone/date semantics unconfirmed.

Coverage is observed unique dates divided by calendar dates in each participant's first-to-last observed span. It does not cover unknown enrolment boundaries and does not turn missing dates into zero heart rate. Aggregate coverage summaries hide identifiers and show unequal follow-up.

In [ ]:
h['period_hours']=h.duration_measure_period_sec/3600
elapsed=h.end_measure_period_sec-h.start_measure_period_sec
ordered=h.sort_values(key)
prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
utc=pd.to_datetime(h.start_measure_period_sec,unit='s',utc=True,errors='coerce')
display(pd.DataFrame([{'nonpositive_duration':int(h.duration_measure_period_sec.le(0).sum()),
 'end_not_after_start':int(elapsed.le(0).sum()),'duration_matches_end_minus_start':int(h.duration_measure_period_sec.eq(elapsed).sum()),
 'overlapping_period_rows':int(ordered.start_measure_period_sec.lt(prior).sum()),
 'unparseable_unix_second_starts':int(utc.isna().sum()),
 'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(h.calendar_date).sum())}]))
display(h.period_hours.value_counts().sort_index().rename_axis('reported_hours').to_frame('records'))
coverage=h.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed_dates=('calendar_date','nunique'))
coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
display(coverage[['observed_dates','span_days','missing_dates_within_span','observed_date_percent']].describe().T)


## 4. Distributions and follow-up weighting
Plot recorded positive values only, with exclusions counted above. The earlier statistics retain every numeric value. Display participant medians only in aggregate. Record-weighted distributions give more influence to participants with more observations; participant medians describe a different weighting. Scores are shown in recorded units, provisionally beats/minute; no clinical thresholds are drawn.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(10,7))
for ax,m in zip(axes.flat,metrics):
    h.loc[h[m].gt(0),m].plot.hist(bins=20,ax=ax)
    ax.set(title=m+' (positive subset)',xlabel='Recorded value; units unverified',ylabel='Records')
plt.tight_layout(); plt.show()
positive=h[metrics].where(h[metrics].gt(0))
display(positive.groupby(h.sdv_user_id).median().describe().T.rename_axis('distribution_of_participant_medians'))
display(h.assign(period_scope=np.where(h.period_hours.eq(24),'exactly 24 hours','other duration')).groupby('period_scope')[metrics].agg(['count','median','min','max']))


## 5. Outer period join: explain unequal record counts
Join on participant/date/start/end with explicit one-to-one validation. Count unmatched keys, compare IDs/group/duration on exact matches, and use a separate participant/date join to distinguish absent dates from different period bounds. A date match alone is not sufficient for an activity/heart-rate comparison.

Describe unmatched activity records using aggregate durations and steps only; never recode their missing heart-rate summary as zero. The data cannot explain why a summary is absent. A difference in table lengths does not establish correspondence until keys are checked.

In [ ]:
joined=None
if valid_key(h,key) and valid_key(tables['activity'],key):
    joined=h.merge(tables['activity'],on=key,how='outer',suffixes=('_hr','_activity'),indicator=True,validate='one_to_one')
    j=joined; both=j._merge.eq('both')
    display(j._merge.value_counts().rename_axis('period_join_status').to_frame('records'))
    agreement=[]
    for field in ['additional_id','sdv_group_id','duration_measure_period_sec']:
        a=j.loc[both,field+'_hr']; b=j.loc[both,field+'_activity']; present=a.notna() & b.notna()
        agreement.append({'field':field,'both_nonnull':int(present.sum()),'equal':int((present & a.eq(b)).sum()),
          'different':int((present & a.ne(b)).sum()),'missing_either':int((~present).sum())})
    display(pd.DataFrame(agreement))
    absent=j._merge.eq('right_only')
    print('Activity-only periods:',int(absent.sum()),'Participants with activity-only periods:',j.loc[absent,'sdv_user_id'].nunique())
    display(j.loc[absent,['steps','duration_measure_period_sec_activity']].describe().T)
    for status in ['both','right_only','left_only']:
        subset=j.loc[j._merge.eq(status)]
        print(status+': exactly-24-hour activity periods:',int(subset.duration_measure_period_sec_activity.eq(86400).sum()))
    date_key=['sdv_user_id','calendar_date']
    if valid_key(h,date_key) and valid_key(tables['activity'],date_key):
        date_join=h[date_key].merge(tables['activity'][date_key],on=date_key,how='outer',indicator=True,validate='one_to_one')
        display(date_join._merge.value_counts().rename_axis('participant_date_join_status').to_frame('records'))
    matched_by_user=j.groupby('sdv_user_id')._merge.agg(lambda x:x.eq('both').sum())
    activity_by_user=tables['activity'].groupby('sdv_user_id').size()
    display((100*matched_by_user/activity_by_user).describe().to_frame('percent_activity_periods_with_exact_HR_match_per_participant'))
else:
    print('Join skipped: resolve null or duplicated period keys first.')


## 6. Average/resting summaries versus recorded steps
For each outcome, require exact period match, agreeing context, nonmissing positive heart-rate value and nonnegative recorded steps. These are operational exclusions, not a clinical definition of validity. Report complete-case counts separately for average and resting outcomes. Repeat for 24-hour windows, then positive-steps 24-hour windows; do not hide recorded zeros.

Pearson is linear association and Spearman rank association. Centering both variables on each participant's subset mean separates within-participant deviations from pooled mean differences. It still weights participants by record count and does not eliminate time trends, selective missingness, overlap or repeated-measure dependence. Participant correlations require at least three observations and variation. Comparing a resting algorithm with activity does not establish its definition or a causal effect.

In [ ]:
results=[]; individual=[]
if joined is not None:
    j=joined
    context=j.sdv_group_id_hr.eq(j.sdv_group_id_activity) & j.duration_measure_period_sec_hr.eq(j.duration_measure_period_sec_activity)
    matched=j._merge.eq('both') & context
    def corr(d,x,y,method='pearson'):
        return d[x].corr(d[y],method=method) if len(d)>=3 and d[x].nunique()>1 and d[y].nunique()>1 else np.nan
    for outcome in ['avg_heart_rate_min','rest_heart_rate_min']:
        eligible=matched & j[outcome].notna() & j[outcome].gt(0) & j.steps.notna() & j.steps.ge(0)
        print(outcome+': matched records excluded for missing/nonpositive HR or missing/negative steps:',int((matched & ~eligible).sum()))
        scopes={'all eligible periods':eligible,'exactly 24 hours':eligible & j.duration_measure_period_sec_hr.eq(86400),
          '24 hours, positive steps':eligible & j.duration_measure_period_sec_hr.eq(86400) & j.steps.gt(0)}
        for label,mask in scopes.items():
            d=j.loc[mask].copy()
            for m in ['steps',outcome]: d[m+'_centered']=d[m]-d.groupby('sdv_user_id')[m].transform('mean')
            results.append({'outcome':outcome,'scope':label,'records':len(d),'participants':d.sdv_user_id.nunique(),
              'pooled_Pearson':corr(d,'steps',outcome),'pooled_Spearman':corr(d,'steps',outcome,'spearman'),
              'within_centered_Pearson':corr(d,'steps_centered',outcome+'_centered')})
            for _,part in d.groupby('sdv_user_id'):
                individual.append({'outcome':outcome,'scope':label,'Pearson':corr(part,'steps',outcome)})
    display(pd.DataFrame(results))
    display(pd.DataFrame(individual).groupby(['outcome','scope']).Pearson.agg(['count','median','min','max']).rename(columns={'count':'participants_with_defined_correlation'}))
    fig,axes=plt.subplots(2,2,figsize=(11,8))
    for row,outcome in enumerate(['avg_heart_rate_min','rest_heart_rate_min']):
        mask=matched & j[outcome].notna() & j[outcome].gt(0) & j.steps.notna() & j.steps.ge(0) & j.duration_measure_period_sec_hr.eq(86400)
        d=j.loc[mask].copy()
        for m in ['steps',outcome]: d[m+'_centered']=d[m]-d.groupby('sdv_user_id')[m].transform('mean')
        axes[row,0].scatter(d.steps,d[outcome],alpha=.35,s=15)
        axes[row,0].set(title=outcome+': pooled 24-hour windows',xlabel='Recorded steps',ylabel='Recorded HR value')
        axes[row,1].scatter(d.steps_centered,d[outcome+'_centered'],alpha=.35,s=15)
        axes[row,1].axhline(0,color='gray',lw=.7); axes[row,1].axvline(0,color='gray',lw=.7)
        axes[row,1].set(title=outcome+': within-participant deviations',xlabel='Steps minus participant mean',ylabel='HR minus participant mean')
    plt.tight_layout(); plt.show()


## Verified findings from this execution
- **961 heart-rate summary records across 8 participants**, versus **987 activity records**. Tested keys are unique/non-null; no exact duplicate rows occur. Minimum, maximum and average each have **one null**, all in the same incomplete triple; resting is non-null. No recorded zero/negative heart-rate values occur.
- All **960 complete triples satisfy min ≤ average ≤ max**. Among 960 records with resting and extrema present, resting is below the reported minimum in **34**, above maximum in **0**. This is an algorithm/window-definition question, not automatically an invalidity or clinical finding.
- Recorded medians are **52 minimum**, **129 maximum**, **74 average** and **56 resting**, in unverified source units. Units and aggregation/resting algorithms require documentation.
- **949 windows are exactly 24 hours**, 12 have other durations. All durations equal end minus start; **two overlaps** occur. Under the Unix-seconds interpretation, UTC start dates do not equal calendar labels; timezone conventions remain unconfirmed. Participants have **15–250 observed dates**, with **93.04%–100% coverage** inside their individual first-to-last observed spans. These are date-coverage diagnostics, not wear-time estimates.
- Exact-period outer join: **961 matches, 26 activity-only periods, no heart-rate-only periods**. Compared record IDs, groups and durations agree in all matches. The participant/date join gives the same 961/26/0 counts, so the unmatched activity periods have absent heart-rate dates rather than merely different bounds. The 26 absent periods involve **5 participants**, include **21 exactly-24-hour windows**, and have median steps **0**, range **0–12,781**. Absence cannot be attributed solely to inactivity or short windows from this evidence. The reason is unknown; missing summaries are never recoded to zero.
- Average versus steps: **960 complete positive-average/nonnegative-steps records**; **948** are exactly 24 hours, with pooled Pearson **0.106**, Spearman **0.098**, and within-participant-centered Pearson **0.387**. Participant-specific 24-hour Pearson correlations range **0.276–0.722** (median **0.458**, eight defined correlations).
- Resting versus steps: **961 eligible records**; **949** are exactly 24 hours, with pooled Pearson **−0.191**, Spearman **−0.139**, and centered Pearson **−0.004**. The different pooled/centered results illustrate how between-participant differences can affect a pooled association. They do not establish mechanisms or causes.

**Takeaway:** Heart-rate summaries cover fewer periods than activity; the complete min/average/max summaries are internally ordered, while resting semantics remain uncertain. Pooled associations can differ substantially from within-participant associations, and complete-case coverage must accompany every result. The sample table remains unreviewed.


## 7. Teacher-facing method and limits
“I checked summary values, ordering and reporting periods, then matched activity by the exact participant/date/period key. I kept absent heart-rate summaries separate from zero values and reported complete-case denominators before comparing pooled and within-participant associations.”

Use the verified results above with their denominators. Positive values are not automatically verified measurements. Resting values need not use the same algorithm or time interval as the other summaries; outside-extrema observations are a question for the owner, not automatically an error. Reporting duration does not prove wear time; missing summaries and unequal follow-up may be selective. Correlation is descriptive and repeated records are not independent participants.

Ask the data owner to confirm units, min/max/average window and weighting, resting algorithm/window, sentinel codes, missing-summary generation rules, timezone/endpoints and wear coverage. This notebook has not reviewed heart_rate_sample. No clinical, causal or predictive conclusion is made. CSV export, RAG and model building remain deferred.